In [ ]:
# This script downloads images from Wikidata based on entity IDs (QIDs) or searches Google Images if no image is found on Wikidata.


import requests
import os
import urllib.parse
import urllib.request
import time
from bs4 import BeautifulSoup
import json
from pathlib import Path
import tqdm

USER_AGENT = "WikidataImageDownloader/1.0 (mailto:<your-email@example.com>)"

class WikidataImageDownloader:
    def __init__(self, user_agent=USER_AGENT):
        self.wikidata_api_url = "https://www.wikidata.org/w/api.php"
        self.headers = {'User-Agent': user_agent}
        self.request_delay = 3  
        
    def search_by_entity_id(self, entity_id: str, download_path: str) -> bool:
        """
        Search Wikidata by entity ID and download image (property P18) if available 
        
        :param entity_id: str, Wikidata entity ID
        :param download_path: str, path where to save the image
        :return: bool, True if image downloaded successfully, False otherwise
        """
        try:
            if not entity_id.startswith('Q'):
                entity_id = 'Q' + entity_id.lstrip('Q')
            
            print(f"Searching for entity ID: {entity_id}")
        
            params = {
                'action': 'wbgetentities',
                'ids': entity_id,
                'format': 'json',
                'props': 'claims'
            }
            
            response = requests.get(self.wikidata_api_url, params=params, headers=self.headers)
            time.sleep(self.request_delay) 
            data = response.json()
            
            if 'entities' not in data or entity_id not in data['entities']:
                print(f"Entity {entity_id} not found")
                return False
            
            entity = data['entities'][entity_id]
            
            if 'claims' in entity and 'P18' in entity['claims']:
                image_claims = entity['claims']['P18']
                if image_claims:
                    image_filename = image_claims[0]['mainsnak']['datavalue']['value']
                    return self._download_wikimedia_image(image_filename, download_path)
            
            print(f"No image found for entity {entity_id}")
            return False
            
        except Exception as e:
            print(f"Error searching by entity ID: {e}")
            return False
    
    def _download_wikimedia_image(self, filename: str, download_path: str) -> bool:
        try:
            filename = filename.replace(' ', '_')
            url = f"https://commons.wikimedia.org/wiki/Special:FilePath/{urllib.parse.quote(filename)}"
            
            print(f"Downloading image: {filename}")
            print(f"URL: {url}")
            
            os.makedirs(os.path.dirname(download_path), exist_ok=True)
            
            req = urllib.request.Request(url, headers=self.headers)
            
            with urllib.request.urlopen(req) as response:
                with open(download_path, 'wb') as f:
                    f.write(response.read())
            
            print(f"Image downloaded successfully to: {download_path}")
            return True
            
        except Exception as e:
            print(f"Error downloading Wikimedia image: {e}")
            return False

    def download_google_image(self, search_query: str, download_path: str) -> bool:
        """
        Search Google Images and download the first valid image found
        :param search_query: str, search query
        :param download_path: str, path where to save the image
        :return: bool, True if image downloaded successfully, False otherwise
        """
        try:
            print(f"Searching Google Images for: {search_query}")
            
            search_url = f"https://www.google.com/search?q={urllib.parse.quote(search_query)}&tbm=isch"
            
            response = requests.get(search_url, headers=self.headers)
            time.sleep(self.request_delay)
            soup = BeautifulSoup(response.content, 'html.parser')

            img_tags = soup.find_all('img')
            
            for img in img_tags:
                img_url = img.get('src') or img.get('data-src')
                if img_url and img_url.startswith('http') and not img_url.startswith('data:'):
                    try:
                        print(f"Attempting to download: {img_url}")
                        os.makedirs(os.path.dirname(download_path), exist_ok=True)
                        
                        req = urllib.request.Request(img_url, headers=self.headers)
                        with urllib.request.urlopen(req) as img_response:
                            with open(download_path, 'wb') as f:
                                f.write(img_response.read())
                        
                        if os.path.getsize(download_path) < 1024:  
                            print(f"Downloaded file is too small, trying next image...")
                            os.remove(download_path)
                            continue
                        
                        print(f"Google image downloaded successfully to: {download_path}")
                        return True
                        
                    except Exception as e:
                        print(f"Failed to download {img_url}: {e}")
                        continue
            
            print("No suitable images found on Google Images")
            return False
            
        except Exception as e:
            print(f"Error with Google Images search: {e}")
            return False


def process_json_items(items_list, output_dir="images"):
    """
    Process a list of JSON items, fetch images from Wikidata or Google, and download them.
    
    :param items_list: List of dictionaries containing item data with 'id' and 'name' fields
    :param output_dir: Directory to save downloaded images
    :return: List of results with download status
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    downloader = WikidataImageDownloader()
    results = []
    
    for idx, item in enumerate(tqdm.tqdm(items_list)):
        qid = item.get('id')
        name = item.get('name', 'unknown')
        domain = item.get('domain', '')
        country = item.get('country', '')
        
        safe_name = "".join(c for c in name if c.isalnum() or c in (' ', '-', '_')).strip()
        safe_name = safe_name.replace(' ', '_')
        
        download_path = os.path.join(output_dir, f"{qid}_{safe_name}.jpg")
        
        if os.path.exists(download_path):
            print(f"Image already exists at {download_path}, skipping download.")
            continue
        
        success = downloader.search_by_entity_id(qid, download_path)
        source = "wikidata" if success else None

        if not success:
            print(f"\nNo Wikidata image found. Searching Google Images...")
            
            search_query = name
            if domain:
                search_query += f" {domain}"
            if country:
                search_query += f" {country}"
            
            success = downloader.download_google_image(search_query, download_path)
            source = "google" if success else None
        
        result = {
            'qid': qid,
            'name': name,
            'domain': domain,
            'country': country,
            'downloaded': success,
            'source': source,
            'local_path': download_path if success else None
        }
        results.append(result)
        
        if success:
            print(f"Successfully downloaded image for {name}")
        else:
            print(f"Failed to download image for {name}")
    
    return results


In [ ]:

file_path = "CUBE-MT/CUBE_CSpace.json"
with open(file_path, 'r', encoding='utf-8') as f:
    items = json.load(f)

output_dir = "<save_dir_path>"
results = process_json_items(items, output_dir)